# **IndoHybrid: TF-IDF and Semantic Search**

**Table of Contents**
1. Install Libraries
2. Config
3. Load Dataset
4. Text Preprocessing
5. Sparse and Dense Indexing (TF-IDF and FAISS)
6. Rank Fusion Implementation
7. Hybrid Search Pipeline
8. Initialization and Benchmarking
9. Evaluation
10. Inference Testing
11. Artifact Serialization and Persistence Test

**1. Install Libraries**

In [55]:
!pip install -q sentence-transformers faiss-cpu scikit-learn datasets joblib

import os
import re
import json
import warnings
import torch
import joblib
import nltk
import numpy as np
import pandas as pd
import faiss

from datasets import disable_progress_bar, load_dataset
from dataclasses import dataclass, asdict
from nltk.corpus import stopwords
from pathlib import Path
from typing import List, Dict, Any, Tuple
from scipy.sparse import save_npz, load_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

In [96]:
nltk.download("stopwords", quiet=True)

disable_progress_bar()
warnings.filterwarnings("ignore")

**2. Config**

In [97]:
@dataclass
class Config:
    embedding_model: str = "intfloat/multilingual-e5-small"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    max_features: int = 50_000
    ngram_range: Tuple[int, int] = (1, 2)
    min_df: int = 1
    max_df: float = 0.95

    candidate_k: int = 50
    top_k: int = 5
    rrf_k: int = 60

    max_query_length: int = 500
    max_document_length: int = 2000

    artifact_dir: str = "./hybrid_search_artifact"

CONFIG = Config()

**3. Load Dataset**

In [98]:
ds = load_dataset("andreaschandra/tydiqa-id", split="train") #source: https://huggingface.co/datasets/andreaschandra/tydiqa-id
df_raw = pd.DataFrame(ds)

str_cols = [c for c in df_raw.columns if isinstance(df_raw[c].iloc[0], str)]
text_col = max(str_cols, key=lambda c: df_raw[c].str.len().mean())

df = df_raw[[text_col]].drop_duplicates().reset_index(drop=True)
df["id"] = [f"DOC_{i+1:05d}" for i in range(len(df))]
df["title"] = "Materi Edukasi"
df["text"] = df.pop(text_col)

In [113]:
def validate_dataset(df_input: pd.DataFrame) -> pd.DataFrame:
    df_clean = df_input.dropna(subset=['text']).copy()
    df_clean['text'] = df_clean['text'].astype(str).str.strip()
    df_clean = df_clean.drop_duplicates(subset=['text']).reset_index(drop=True)
    df_clean['id'] = [f"DOC_{i+1:05d}" for i in range(len(df_clean))]

    return df_clean

df = validate_dataset(df)

**4. Text Preprocessing**

In [114]:
class IndonesianPreprocessor:
    def __init__(self):
        self.stopwords = set(stopwords.words("indonesian"))

    def normalize(self, text: Any, max_length: int = 2000) -> str:
        if not isinstance(text, str):
            text = str(text) if pd.notna(text) else ""
        text = text.strip().lower()
        if not text:
            return ""
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z0-9\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text[:max_length]

    def for_tfidf(self, text: Any, max_length: int = 2000) -> str:
        clean_text = self.normalize(text, max_length)
        if not clean_text:
            return ""
        words = [w for w in clean_text.split() if w not in self.stopwords]
        return " ".join(words) if words else clean_text

    def get_full_sentence_snippet(self, text: str, max_chars: int = 300) -> str:
        clean_text = text.replace("\n", " ").strip()
        sentences = re.split(r"(?<=[.!?])\s+", clean_text)

        snippet = []
        current_length = 0

        for sentence in sentences:
            if not snippet or (current_length + len(sentence) <= max_chars):
                snippet.append(sentence)
                current_length += len(sentence)
            else:
                break

        return " ".join(snippet)

**5. Sparse and Dense Indexing (TF-IDF and FAISS)**

In [117]:
class TfidfIndex:
    def __init__(self, config: Config):
        self.config = config
        self.vectorizer = TfidfVectorizer(
            max_features=config.max_features,
            ngram_range=config.ngram_range,
            min_df=config.min_df,
            max_df=config.max_df,
            sublinear_tf=True,
            smooth_idf=True,
            token_pattern=r"(?u)\b\w+\b"
        )
        self.matrix = None

    def build(self, texts: List[str]):
        self.matrix = self.vectorizer.fit_transform(texts)

    def search(self, query_text: str, k: int) -> List[Tuple[int, float]]:
        if not query_text or self.matrix is None:
            return []
        query_vec = self.vectorizer.transform([query_text])

        sparse_scores = self.matrix @ query_vec.T
        non_zero_indices = sparse_scores.nonzero()[0]

        if len(non_zero_indices) == 0:
            return []

        scores = sparse_scores.data
        top_k_indices = np.argsort(-scores)[:k]

        return [(int(non_zero_indices[idx]), float(scores[idx])) for idx in top_k_indices]


class EmbeddingIndex:
    def __init__(self, config: Config):
        self.config = config
        self.model = SentenceTransformer(config.embedding_model, device=config.device)
        self.index = None
        self.dimension = None

    def build(self, texts: List[str]):
        formatted_texts = [f"passage: {t}" for t in texts]
        embeddings = self.model.encode(
            formatted_texts,
            batch_size=128 if self.config.device == "cuda" else 32,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        self.dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(self.dimension)
        self.index.add(embeddings)

    def search(self, query_text: str, k: int) -> List[Tuple[int, float]]:
        if not query_text or self.index is None:
            return []

        formatted_query = f"query: {query_text}"
        query_vec = self.model.encode(
            [formatted_query],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        k = min(k, self.index.ntotal)
        if k == 0:
            return []

        scores, indices = self.index.search(query_vec, k)
        return [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0]) if idx >= 0]

**6. Rank Fusion Implementation**

In [118]:
def reciprocal_rank_fusion(
    rankings: List[List[Tuple[int, float]]],
    rrf_k: int = 60,
    top_k: int = 5
) -> List[Tuple[int, float]]:
    fused_scores = {}
    for ranking in rankings:
        for rank, (doc_idx, _) in enumerate(ranking, start=1):
            fused_scores[doc_idx] = fused_scores.get(doc_idx, 0.0) + 1.0 / (rrf_k + rank)

    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

**7. Hybrid Search Pipeline**

In [119]:
class HybridSearch:
    def __init__(self, config: Config):
        self.config = config
        self.preprocessor = IndonesianPreprocessor()
        self.documents = []
        self.tfidf = TfidfIndex(config)
        self.embedding = EmbeddingIndex(config)
        self.ready = False

    def validate_input(self, df: pd.DataFrame):
        required = {"id", "title", "text"}
        if not required.issubset(df.columns):
            raise ValueError(f"Required column not found: {required - set(df.columns)}")
        if df.empty:
            raise ValueError("The dataset must not be empty.")
        if df["id"].duplicated().any():
            raise ValueError("The document ID must be unique.")

    def build(self, df: pd.DataFrame):
        self.validate_input(df)

        self.documents = df[["id", "title", "text"]].fillna("").to_dict("records")
        full_texts = [f"{doc['title']}. {doc['text']}" for doc in self.documents]

        tfidf_texts = [self.preprocessor.for_tfidf(t, self.config.max_document_length) for t in full_texts]

        self.tfidf.build(tfidf_texts)
        self.embedding.build(full_texts)

        self.ready = True
        return self

    def search(self, query: str, top_k: int = None, candidate_k: int = None) -> List[Dict[str, Any]]:
        if not self.ready:
            raise RuntimeError("The engine has not been built yet.")

        top_k = top_k or self.config.top_k
        candidate_k = candidate_k or self.config.candidate_k

        tfidf_query = self.preprocessor.for_tfidf(query, self.config.max_query_length)

        if not tfidf_query and not query.strip():
            return []

        lexical_results = self.tfidf.search(tfidf_query, candidate_k)
        semantic_results = self.embedding.search(query, candidate_k)

        fused = reciprocal_rank_fusion(
            [lexical_results, semantic_results],
            rrf_k=self.config.rrf_k,
            top_k=top_k
        )

        tfidf_map = dict(lexical_results)
        semantic_map = dict(semantic_results)

        results = []
        for rank, (doc_idx, rrf_score) in enumerate(fused, start=1):
            doc = self.documents[doc_idx]
            results.append({
                "rank": rank,
                "id": doc["id"],
                "title": doc["title"],
                "text": doc["text"],
                "rrf_score": round(rrf_score, 6),
                "tfidf_score": round(tfidf_map.get(doc_idx, 0.0), 6),
                "semantic_score": round(semantic_map.get(doc_idx, 0.0), 6),
            })

        return results

    def save(self, path: str = None) -> str:
        save_path = Path(path or self.config.artifact_dir).resolve()
        save_path.mkdir(parents=True, exist_ok=True)

        joblib.dump(self.tfidf.vectorizer, save_path / "tfidf_vectorizer.joblib")
        save_npz(save_path / "tfidf_matrix.npz", self.tfidf.matrix)
        faiss.write_index(self.embedding.index, str(save_path / "faiss.index"))

        with open(save_path / "documents.json", "w", encoding="utf-8") as f:
            json.dump(self.documents, f, ensure_ascii=False, indent=2)

        metadata = {"config": asdict(self.config)}
        with open(save_path / "metadata.json", "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

        return str(save_path)

    def load(self, path: str = None):
        load_path = Path(path or self.config.artifact_dir).resolve()
        if not load_path.exists():
            raise FileNotFoundError(f"Artifact not found in: {load_path}")

        with open(load_path / "metadata.json", "r", encoding="utf-8") as f:
            metadata = json.load(f)
            if "config" in metadata:
                for k, v in metadata["config"].items():
                    if hasattr(self.config, k):
                        setattr(self.config, k, tuple(v) if isinstance(v, list) else v)

        self.tfidf.vectorizer = joblib.load(load_path / "tfidf_vectorizer.joblib")
        self.tfidf.matrix = load_npz(load_path / "tfidf_matrix.npz")
        self.embedding.index = faiss.read_index(str(load_path / "faiss.index"))
        self.embedding.dimension = self.embedding.index.d

        with open(load_path / "documents.json", "r", encoding="utf-8") as f:
            self.documents = json.load(f)

        self.ready = True
        return self

**8. Initialization and Benchmarking**

In [120]:
searcher = HybridSearch(CONFIG)
searcher.build(df)

def run_benchmark(searcher, eval_df, k=10, sample_size=50):
    stats = {
        "hybrid": {"hits": 0, "rr": []},
        "lexical": {"hits": 0, "rr": []},
        "semantic": {"hits": 0, "rr": []}
    }

    sample_eval = eval_df.sample(min(sample_size, len(eval_df)), random_state=42)
    doc_text_to_id = {doc["text"].strip(): doc["id"] for doc in searcher.documents}

    valid_eval_count = 0

    for _, row in tqdm(sample_eval.iterrows(), total=len(sample_eval), desc="Benchmarking Engine"):
        query = row.get("question_text", "")
        actual_text = row.get("document_plaintext", "").strip()
        expected_id = doc_text_to_id.get(actual_text)

        if not query or not expected_id:
            continue

        valid_eval_count += 1

        tfidf_query = searcher.preprocessor.for_tfidf(query, searcher.config.max_query_length)

        lex_res = searcher.tfidf.search(tfidf_query, k)
        sem_res = searcher.embedding.search(query, k)
        hyb_res = searcher.search(query, top_k=k)

        def check_hits(results, target_id, key):
            found_at = -1
            for i, res in enumerate(results):
                res_id = res["id"] if isinstance(res, dict) else searcher.documents[res[0]]["id"]
                if res_id == target_id:
                    found_at = i + 1
                    break

            if found_at > 0:
                stats[key]["hits"] += 1
                stats[key]["rr"].append(1.0 / found_at)
            else:
                stats[key]["rr"].append(0.0)

        check_hits(hyb_res, expected_id, "hybrid")
        check_hits(lex_res, expected_id, "lexical")
        check_hits(sem_res, expected_id, "semantic")

    return stats, valid_eval_count

benchmark_stats, total_valid_samples = run_benchmark(searcher, df_raw, k=10, sample_size=50)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Benchmarking Engine:   0%|          | 0/50 [00:00<?, ?it/s]

**9. Evaluation**

In [121]:
def calculate_evaluation_metrics(stats, total_samples):
    total = total_samples if total_samples > 0 else 1
    results = {}
    for m in ["hybrid", "lexical", "semantic"]:
        results[m] = {
            "hit_rate": stats[m]["hits"] / total,
            "mrr": float(np.mean(stats[m]["rr"])) if len(stats[m]["rr"]) > 0 else 0.0
        }
    return results

eval_results = calculate_evaluation_metrics(benchmark_stats, total_valid_samples)

print("\n" + "="*45)
print(f" {'Metode':<15} | {'Hit Rate':<10} | {'MRR':<8}")
print("-" * 45)
for mode in ["lexical", "semantic", "hybrid"]:
    name = mode.capitalize() if mode != "hybrid" else "HYBRID (RRF)"
    print(f" {name:<15} | {eval_results[mode]['hit_rate']:>9.2%} | {eval_results[mode]['mrr']:>8.4f}")
print("="*45)


 Metode          | Hit Rate   | MRR     
---------------------------------------------
 Lexical         |    84.00% |   0.6199
 Semantic        |    90.00% |   0.6947
 HYBRID (RRF)    |    88.00% |   0.6865


**10. Inference Testing**

In [134]:
query = "Beri saya informasi hal-hal yang berkaitan dengan Palembang!"
results = searcher.search(query, top_k=3)

print(f"QUERY: '{query}'\n" + " ")
for res in results:
    snippet = searcher.preprocessor.get_full_sentence_snippet(
        res["text"], max_chars=300
    )
    print(f"[{res['rank']}] ID: {res['id']} | RRF: {res['rrf_score']:.4f}")
    print(f"Result: {snippet}")

QUERY: 'Beri saya informasi hal-hal yang berkaitan dengan Palembang!'
 
[1] ID: DOC_03047 | RRF: 0.0304
Result: Kota Palembang adalah ibu kota provinsi Sumatera Selatan. Palembang adalah kota terbesar kedua di Sumatera setelah Medan. Kota Palembang memiliki luas wilayah 358,55km²[1] yang dihuni 1.573.898 jiwa (2018) dengan kepadatan penduduk 4.800 per km².
[2] ID: DOC_05586 | RRF: 0.0303
Result: Wayang Kulit Palembang adalah sebuah bentuk pewayangan dengan visi dan versi dari masyarakat Palembang itu sendiri, jenis kesenian ini diperkirakan tumbuh pada sekitar abad 19 (tahun 1800an) pada masa pemerintahan Arya Damar.
[3] ID: DOC_00878 | RRF: 0.0293
Result: Kesultanan Palembang Darussalam adalah suatu kerajaan Islam di Indonesia yang berlokasi di sekitar kota Palembang, Sumatera Selatan sekarang.


**11. Artifact Serialization and Persistence Test**

In [133]:
artifact_path = searcher.save()

reloaded_searcher = HybridSearch(CONFIG)
reloaded_searcher.load(artifact_path)

api_test = reloaded_searcher.search("Beri saya informasi hal-hal yang berkaitan dengan Palembang!", top_k=1)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]